**MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

**Curso: TC4034.10- Análisis de grandes volúmenes de datos**

Tecnológico de Monterrey

Dr. Iván Olmos Pineda

Integrantes del equipo 31:
* Leonardo Javier Nava Castellanos 					A01750595
* Alejandro Jesús Mondragón Jiménez  					A01795837
* Eva Denisse Vargas Sosa  						A01377098
* Gloria María Campos García 						A01422345

**Proyecto** |  Base de Datos de Big Data

##  Sistema de Recuperación de submuestras de NYC Taxi

---

### Descripción del cuaderno
 Este cuaderno tiene como objetivo procesar la base de datos D (NYC Yellow Taxi Trip Data) para filtrar y extraer submuestras de registros específicos basados en ciertos criterios de movilidad urbana.

### Base de Datos (D)
La base de datos (D) corresponde al conjunto de datos de NYC Yellow Taxi Trip Data obtenida de kaggle [link](https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data?resource=download), específicamente considerando los periodos de enero de 2015 y marzo de 2016. En el que cada registro representa un viaje individual con información temporal, geográfica y operativa.

### Reglas de Particionamiento
 Se utilizan dos variables derivadas para segmentar la población:

1. time_period: Clasifica el inicio del viaje en Madrugada (00:00-05:59), Mañana (06:00-11:59), Tarde (12:00-17:59) y Noche (18:00-23:59).

2. distance_category: Clasifica los viajes por su longitud en Corta (menor a 2 millas), Media (entre 2 y 10 millas) y Larga (mayor a 10 millas).

Por lo que este código permite extraer registros que cumplan cruces específicos, como "Madrugada + Corta" (viajes breves nocturnos) o "Mañana + Larga" (traslados extensos en horas de alta movilidad).


### 1. Configuración del entorno

In [10]:
# Librerias 
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
findspark.init(r"C:\\Users\\gloca\\OneDrive\\Documents\\spark-4.1.1-bin-hadoop3\\spark-4.1.1-bin-hadoop3")

In [5]:
#Configuración de la sesión 
spark = SparkSession.builder \
    .appName("Recuperacion_Particiones_Taxi") \
    .getOrCreate()

### 2. Cargar los datos originales y realizar limpieza

In [11]:
def read_and_clean(path, rename_ratecode=False):
    # Cargamos con inferSchema=True para asegurar tipos correctos en el filtrado
    df = spark.read.csv(path, header=True, sep=",", inferSchema=True)
    return df 

In [12]:
path_base = "archive\\"
df1 = read_and_clean(path_base + "yellow_tripdata_2015-01.csv")
df2 = read_and_clean(path_base + "yellow_tripdata_2016-01.csv")
df3 = read_and_clean(path_base + "yellow_tripdata_2016-02.csv")
df4 = read_and_clean(path_base + "yellow_tripdata_2016-03.csv")

In [ ]:
# Unión de la base de datos completa D (Población total de aprox 47.2M registros)
df_D = df1.unionByName(df2).unionByName(df3).unionByName(df4)
print(f"Base de datos D cargada. Total de registros: {df_D.count()}")

Base de datos D cargada. Total de registros: 47248845


### 3. Reglas de las submuestras

In [14]:
df_reglas = df_D.withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("time_period", 
        F.when((F.col("pickup_hour") >= 0) & (F.col("pickup_hour") < 6), "Madrugada")
        .when((F.col("pickup_hour") >= 6) & (F.col("pickup_hour") < 12), "Manana")
        .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") < 18), "Tarde")
        .otherwise("Noche")) \
    .withColumn("distance_category", 
        F.when(F.col("trip_distance") < 2, "Corta")
        .when((F.col("trip_distance") >= 2) & (F.col("trip_distance") < 10), "Media")
        .otherwise("Larga"))

### 4. Extracción de submuestras

In [15]:
# Función para filtrar y recuperar registros según las reglas definidas
def recuperar_instancias(periodo, distancia):
    print(f"--- Registros para: {periodo} y Distancia {distancia} ---")
    
    # Filtrado según la combinación
    particion = df_reglas.filter(
        (F.col("time_period") == periodo) & 
        (F.col("distance_category") == distancia)
    )
    
    conteo = particion.count()
    print(f"Total de instancias que cumplen la regla: {conteo}")
    
    # Extracción de muestra de prueba (10% de la partición para verificación)
    return particion.sample(withReplacement=False, fraction=0.1, seed=42)

### 5. Pruebas de funcionamiento

In [17]:
# Ejemplo 1: Regla Madrugada + Corta (Viajes nocturnos breves)
muestra_madrugada_corta = recuperar_instancias("Madrugada", "Corta")
muestra_madrugada_corta.select("tpep_pickup_datetime", "trip_distance", "time_period", "distance_category").show(5)

# Ejemplo 2: Regla Manana + Larga (Viajes extensos en hora pico)
muestra_manana_larga = recuperar_instancias("Manana", "Larga")
muestra_manana_larga.select("tpep_pickup_datetime", "trip_distance", "time_period", "distance_category").show(5)

print("Proceso de validación de recuperación finalizado.")

--- Registros para: Madrugada y Distancia Corta ---
Total de instancias que cumplen la regla: 2471209
+--------------------+-------------+-----------+-----------------+
|tpep_pickup_datetime|trip_distance|time_period|distance_category|
+--------------------+-------------+-----------+-----------------+
| 2015-01-25 00:13:09|         1.14|  Madrugada|            Corta|
| 2015-01-01 01:08:57|          1.2|  Madrugada|            Corta|
| 2015-01-01 05:29:48|         1.29|  Madrugada|            Corta|
| 2015-01-21 00:16:33|          1.5|  Madrugada|            Corta|
| 2015-01-21 00:53:16|          1.6|  Madrugada|            Corta|
+--------------------+-------------+-----------+-----------------+
only showing top 5 rows
--- Registros para: Manana y Distancia Larga ---
Total de instancias que cumplen la regla: 562994
+--------------------+-------------+-----------+-----------------+
|tpep_pickup_datetime|trip_distance|time_period|distance_category|
+--------------------+-------------+---